# Tutorial: run TRIM and analyse the output with DataFrames

Set up SRIM once with `pysrim2 install` and check it with `pysrim2 doctor`
(see the [installation guide](https://pysrim2.readthedocs.io/en/latest/installation.html)). The notebook then finds SRIM
by itself, natively or in the container; no path is needed. SRIM-2013 only, **not** 2018.
SRIM is (c) SRIM.com (James F. Ziegler), free for non-commercial use only; see `LEGAL.rtf` next to your SRIM copy.

We will

- run a quick TRIM calculation,
- turn every output table into a `pandas.DataFrame` (`.df`, `Results.to_dataframes()`),
- summarise where the ions stopped with `implant_summary`,
- export the tables to Parquet.

See the [tutorial](https://pysrim2.readthedocs.io/en/latest/tutorial.html) and the [API reference](https://pysrim2.readthedocs.io/en/latest/source/modules.html)
for the column definitions (names carry SRIM's units: `depth_A`, `energy_eV`, ...).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from srim import TRIM, Ion, Layer, Target
from srim.analysis import implant_summary
from srim.output import Results

OUT = Path("analysis-output")
OUT.mkdir(exist_ok=True)

## Run a calculation

Concepts: a list of `Layer`s forms a `Target`; a `Layer` is a dict of elements with a
density [g/cm3] and a width [Angstrom]; an `Ion` has an energy in eV.

In [ ]:
ion = Ion("Ni", energy=3.0e6)  # 3 MeV

layer = Layer(
    {"Ni": {"stoich": 1.0, "E_d": 30.0, "lattice": 0.0, "surface": 3.0}},
    density=8.9,
    width=20000.0,  # 2 um
)
target = Target([layer])

# 25 ions, quick Kinchin-Pease; ranges=1 also writes RANGE_3D.txt (final positions).
# run() uses the SRIM installed by `pysrim2 install`.
trim = TRIM(target, ion, number_ions=25, calculation=1, ranges=1, random_seed=12345)
results = trim.run()

## Every table is a DataFrame

`results.ioniz`, `.vacancy`, `.phonons`, `.range`, `.etorecoils` (and `.novac` for full
cascades) keep their array attributes; `.df` gives the same numbers as a table whose first
column is `depth_A`.

In [ ]:
results.vacancy.df.head()

In [ ]:
tables = results.to_dataframes()  # dict: name -> DataFrame
{name: df.shape for name, df in tables.items()}

In [ ]:
tables["range3d"].head()  # one row per ion: ion, depth_A, y_A, z_A

## Where did the ions stop?

`implant_summary` uses SRIM's definitions (the numbers in the `RANGE.txt` header):
projected range and straggling along the depth, and lateral and radial straggling.

In [ ]:
summary = implant_summary(results.range3d.df)
summary

In [ ]:
results.range.summary  # the same quantities as SRIM printed them in RANGE.txt

## Damage, ionisation and vacancies versus depth

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ph = results.phonons.df
axes[0].plot(ph["depth_A"], ph["ions_phonons_per_A_per_ion"], label="from ions")
axes[0].plot(ph["depth_A"], ph["recoils_phonons_per_A_per_ion"], label="from recoils")
axes[0].set(title="Phonons", xlabel="Depth [A]", ylabel="phonons / (A ion)")
axes[0].legend()

io = results.ioniz.df
axes[1].plot(io["depth_A"], io["ions_eV_per_A_per_ion"], label="from ions")
axes[1].plot(io["depth_A"], io["recoils_eV_per_A_per_ion"], label="from recoils")
axes[1].set(title="Ionisation", xlabel="Depth [A]", ylabel="eV / (A ion)")
axes[1].legend()

vac = results.vacancy.df
total = vac["knock_ons_per_A_per_ion"] + vac.filter(like="vacancies_").sum(axis=1)
axes[2].plot(vac["depth_A"], total)
axes[2].set(title="Vacancies", xlabel="Depth [A]", ylabel="vacancies / (A ion)")
fig.tight_layout()
plt.show()

## Export

`to_parquet` needs `pip install "pysrim2[parquet]"`; `to_hdf5` needs `pysrim2[hdf5]`;
`to_csv` needs nothing. Each file carries the table's metadata and the run spec.

In [ ]:
results.to_parquet(OUT / "parquet")
sorted(p.name for p in (OUT / "parquet").iterdir())

## Keep SRIM's raw files and read them back later

SRIM overwrites its output with every run. Copy what you need, and `Results(folder)`
parses a folder again later (`run.yaml`, written by `run()`, is copied too).

In [ ]:
(OUT / "raw").mkdir(exist_ok=True)
TRIM.copy_output_files(results.directory, OUT / "raw")
again = Results(OUT / "raw")
print(again.ioniz.num_ions, "ions")
sorted(p.name for p in (OUT / "raw").iterdir())